# Without Eve 

In [1]:
import random
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

n_qubits = 10

# Step 1 - Alice chooses bits and bases
alice_bits = [random.randint(0,1) for _ in range(n_qubits)]
alice_bases = [random.choice(['Z','X']) for _ in range(n_qubits)]

# Step 2 - Bob chooses measurement bases
bob_bases = [random.choice(['Z','X']) for _ in range(n_qubits)]

bob_results = []
backend = Aer.get_backend('qasm_simulator')

for i in range(n_qubits):
    qc = QuantumCircuit(1,1)
    
    # Alice's preparation
    if alice_bases[i] == 'Z':
        if alice_bits[i] == 1:
            qc.x(0)
    else:  # X basis
        if alice_bits[i] == 0:
            qc.h(0)   # |+>
        else:
            qc.x(0)
            qc.h(0)   # |->
    
    # Bob's measurement basis
    if bob_bases[i] == 'X':
        qc.h(0)
    
    qc.measure(0,0)
    
    new_cirq = transpile(qc, backend)
    job = backend.run(new_cirq, shots=1)
    
    measured_bit = int(list(job.result().get_counts().keys())[0])
    bob_results.append(measured_bit)

# ----------- Sifting the keys (AFTER collecting results) -----------
sifted_alice = []
sifted_bob = []

print("\n--- Qubit Results ---")
for i in range(n_qubits):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])
        print(f"{i}: Basis match -> Alice={alice_bits[i]}, Bob={bob_results[i]} ({alice_bases[i]})")
    else:
        print(f"{i}: Basis mismatch -> Discarded (Alice={alice_bases[i]}, Bob={bob_bases[i]})")

print("\nAlice sifted key:", sifted_alice)
print("Bob sifted key:  ", sifted_bob)

# Compute QBER
if len(sifted_alice) > 0:
    errors = sum(a != b for a,b in zip(sifted_alice, sifted_bob))
    qber = errors / len(sifted_alice)
    print(f"\nQBER: {qber:.2%}")
else:
    print("\nNo sifted bits -> QBER undefined")



--- Qubit Results ---
0: Basis match -> Alice=1, Bob=1 (X)
1: Basis match -> Alice=1, Bob=1 (X)
2: Basis match -> Alice=1, Bob=1 (X)
3: Basis mismatch -> Discarded (Alice=Z, Bob=X)
4: Basis mismatch -> Discarded (Alice=Z, Bob=X)
5: Basis match -> Alice=1, Bob=1 (X)
6: Basis match -> Alice=1, Bob=1 (X)
7: Basis mismatch -> Discarded (Alice=X, Bob=Z)
8: Basis mismatch -> Discarded (Alice=Z, Bob=X)
9: Basis mismatch -> Discarded (Alice=X, Bob=Z)

Alice sifted key: [1, 1, 1, 1, 1]
Bob sifted key:   [1, 1, 1, 1, 1]

QBER: 0.00%


# With Eve

In [4]:
import random
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

n_qubits = 10

# Random basis choices
alice_bits = [random.randint(0,1) for _ in range(n_qubits)]
alice_bases = [random.choice(['Z','X']) for _ in range(n_qubits)]
bob_bases   = [random.choice(['Z','X']) for _ in range(n_qubits)]
eve_bases   = [random.choice(['Z','X']) for _ in range(n_qubits)]  # <-- Eve

backend = Aer.get_backend('qasm_simulator')
bob_results = []

print("\n--- Alice → Eve → Bob Transmission ---")

for i in range(n_qubits):
    # ==============================
    # Alice's Preparation
    # ==============================
    qc = QuantumCircuit(1,1)
    if alice_bases[i] == 'Z':
        if alice_bits[i] == 1:
            qc.x(0)
    else:  # X basis
        if alice_bits[i] == 0:
            qc.h(0)
        else:
            qc.x(0)
            qc.h(0)

    # ==============================
    # Eve's Intercept & Measure
    # ==============================
    eve_qc = qc.copy()
    if eve_bases[i] == 'X':
        eve_qc.h(0)
    eve_qc.measure(0,0)

    eve_circ = transpile(eve_qc, backend)
    eve_job = backend.run(eve_circ, shots=1)
    eve_measured_bit = int(list(eve_job.result().get_counts().keys())[0])

    # ==============================
    # Eve Resends New Qubit to Bob
    # ==============================
    resend_qc = QuantumCircuit(1,1)
    if eve_bases[i] == 'Z':
        if eve_measured_bit == 1:
            resend_qc.x(0)
    else:  # X basis
        if eve_measured_bit == 0:
            resend_qc.h(0)
        else:
            resend_qc.x(0)
            resend_qc.h(0)

    # ==============================
    # Bob’s Measurement
    # ==============================
    if bob_bases[i] == 'X':
        resend_qc.h(0)
    resend_qc.measure(0,0)

    bob_circ = transpile(resend_qc, backend)
    job = backend.run(bob_circ, shots=1)
    bob_bit = int(list(job.result().get_counts().keys())[0])
    bob_results.append(bob_bit)

# ======================================
# Sifting (remove mismatched bases)
# ======================================
sifted_alice = []
sifted_bob = []

print("\n--- Basis Check & Key Sifting ---")
for i in range(n_qubits):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])
        print(f"{i}: Match ({alice_bases[i]}) → Alice={alice_bits[i]}, Bob={bob_results[i]}")
    else:
        print(f"{i}: Mismatch (Alice={alice_bases[i]}, Bob={bob_bases[i]}) → Discarded")

# ======================================
# QBER Calculation
# ======================================
print("\nSifted keys:")
print("Alice:", sifted_alice)
print("Bob  :", sifted_bob)

if len(sifted_alice) > 0:
    errors = sum(a != b for a,b in zip(sifted_alice, sifted_bob))
    qber = errors / len(sifted_alice)
    print(f"\nQBER (with Eve): {qber:.2%}")
else:
    print("\nNo sifted bits → QBER undefined")



--- Alice → Eve → Bob Transmission ---

--- Basis Check & Key Sifting ---
0: Mismatch (Alice=Z, Bob=X) → Discarded
1: Mismatch (Alice=X, Bob=Z) → Discarded
2: Match (X) → Alice=0, Bob=1
3: Mismatch (Alice=Z, Bob=X) → Discarded
4: Match (X) → Alice=1, Bob=1
5: Match (X) → Alice=1, Bob=1
6: Mismatch (Alice=Z, Bob=X) → Discarded
7: Match (Z) → Alice=0, Bob=1
8: Match (X) → Alice=1, Bob=1
9: Mismatch (Alice=X, Bob=Z) → Discarded

Sifted keys:
Alice: [0, 1, 1, 0, 1]
Bob  : [1, 1, 1, 1, 1]

QBER (with Eve): 40.00%
